# FFT
Lê um arquivo PDM gravado pelo ESP32 (com ou sem cabeçalho) e plota o gráfico da FFT em escala semilogarítimica.


In [46]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import butter, filtfilt

def fft(filename, fs, bitrate):
    #leitura do arquivo
    if bitrate == 16:
        data = np.fromfile(filename, dtype=np.uint16)
    if bitrate == 32:
        data = np.fromfile(filename, dtype=np.uint32)
    data = data[len(data)//2:]
    data = data.byteswap()
    data2 = data.view(np.uint8)
    bits = np.unpackbits(data2) #bytes para bits
    pdm_signal = 2 * bits.astype(np.float64) - 1 #transformar pra bipolar
    
    # calcular fft
    N = len(pdm_signal)
    spec = np.fft.rfft(pdm_signal)
    freq = np.fft.rfftfreq(N, 1/fs)
    return spec,freq

def plot(spec, freq):
    # plot
    fig, ax = plt.subplots()
    ax.semilogx(freq, 20*np.log10(spec), linewidth=0.5)
    ax.set_xlabel('Frequência (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title('Espectro de Frequência (FFT)')
    plt.grid(True, which="both", linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_compare(sp1, sp2, f1, f2):
    # plot
    fig, (ax1,ax2) = plt.subplots(2,1, figsize=(4,7))
    ax1.semilogx(f1, 20*np.log10(sp1), linewidth=0.5)
    ax1.set_xlabel('Frequência (Hz)')
    ax1.set_ylabel('Magnitude (dB)')
    ax1.set_title('FFT - 1')
    ax1.grid(True, which="both", linestyle='--', alpha=0.3)
    ax2.semilogx(f2, 20*np.log10(sp2), linewidth=0.5)
    ax2.set_xlabel('Frequência (Hz)')
    ax2.set_ylabel('Magnitude (dB)')
    ax2.set_title('FFT - 2')
    ax2.grid(True, which="both", linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.io import wavfile

def fftplot_filtered_wav(filename, fs, bitrate, fc, order, decim, output_wav="output.wav"):
    # 1. Leitura e Conversão PDM
    dtype = np.uint16 if bitrate == 16 else np.uint32
    data = np.fromfile(filename, dtype=dtype)
    data = data[len(data)//8:].byteswap()
    data2 = data.view(np.uint8)
    bits = np.unpackbits(data2)
    pdm_signal = 2.0 * bits.astype(np.float64) - 1.0

    # 2. Filtragem
    # Correção da normalização: Wn = fc / (fs/2)
    norm_fc = fc / (fs * 0.5)
    b, a = butter(N=order, Wn=norm_fc, btype='low')
    filtered_data = filtfilt(b, a, pdm_signal)
    
    # 3. Decimação
    fs_decim = int(fs / decim)
    pcm_signal = filtered_data[::decim]
    
    # --- SALVAMENTO DO ARQUIVO DE ÁUDIO ---
    # Normalização para evitar clipping (deixa um headroom de 10%)
    max_val = np.max(np.abs(pcm_signal))
    if max_val > 0:
        audio_normalized = pcm_signal / max_val * 0.9
    else:
        audio_normalized = pcm_signal

    # Conversão para Int16 (PCM 16-bit)
    audio_int16 = (audio_normalized * 32767).astype(np.int16)
    
    # Escrita do arquivo WAV
    wavfile.write(output_wav, fs_decim, audio_int16)
    print(f"Áudio salvo com sucesso: {output_wav}")
    # --------------------------------------
    
    # 4. Cálculo FFT (usando magnitude absoluta)
    N = len(pcm_signal)
    spec = np.abs(np.fft.rfft(pcm_signal))
    spec = (spec)^2 / (fs*N)
    freq = np.fft.rfftfreq(N, 1/fs_decim)

    # 5. Plotagem
    fig, ax = plt.subplots()
    ax.semilogx(freq, 20 * np.log10(spec + 1e-12), linewidth=0.5)
    ax.set_xlabel('Frequência (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(f'Espectro de Frequência - Fs Final: {fs_decim} Hz')
    plt.grid(True, which="both", linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [48]:
%matplotlib
audio ='data/13-03-testes-clock/16-2-75000hz-1.RAW'
audio2 ='data/13-03-testes-clock/16-2-75000hz-2.RAW'
au3 ='data/13-03-testes-clock/32-2-75000hz_5.RAW'
au4 ='data/13-03-testes-clock/32-2-75000hz_6.RAW'

s1, f1 = fft(audio, 2.4e6, 16)
s2, f2 = fft(audio2, 2.4e6, 16)
s3, f3 = fft(au3, 2.4e6, 32) 
s4, f4 =fft(au4, 2.4e6, 32)

plot_compare(s1,s2,f1,f2)
plot_compare(s3,s4,f3,f4)


Using matplotlib backend: TkAgg


/usr/lib/python3/dist-packages/matplotlib/cbook/__init__.py:1369: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)
